# Inference and Latency

Measures per-image inference latency for early-warning deployment evaluation.

In [ ]:
from pathlib import Path
import sys, time
import numpy as np

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from ultralytics import YOLO
from weapon_threat_detection.artifacts import utc_timestamp, write_json
from weapon_threat_detection.device import select_device

device = select_device()
weights_path = ROOT / 'runs' / 'REPLACE_WITH_RUN' / 'weights' / 'best.pt'
images = list((ROOT / 'processed' / 'yolo_v1' / 'test' / 'images').glob('*'))[:100]
if not weights_path.exists(): raise FileNotFoundError(f'Update weights_path: {weights_path}')
model = YOLO(weights_path); model.predict(images[0], device=device.name, verbose=False)
latencies_ms = []
for image in images:
    started = time.perf_counter(); model.predict(image, device=device.name, verbose=False); latencies_ms.append((time.perf_counter() - started) * 1000)
summary = {'device': device.__dict__, 'samples': len(latencies_ms), 'mean_ms': float(np.mean(latencies_ms)), 'p50_ms': float(np.percentile(latencies_ms, 50)), 'p95_ms': float(np.percentile(latencies_ms, 95))}
report = write_json(ROOT / 'reports' / f'inference_latency_{utc_timestamp()}.json', summary)
print(summary); print(report)